In [1]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings
import re  # ✅ 텍스트 정규화를 위한 정규식 사용

warnings.filterwarnings("ignore")

from pycaret.regression import *  # setup, compare_models, create_model, tune_model, plot_model, predict_model

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge  # ✅ 3순위: Ridge baseline용


class MercariPyCaretAnalyzer:
    """
    Mercari Price Suggestion Challenge용 PyCaret 분석기

    주요 기능:
    1) 데이터 로딩 및 전처리
       - price 필터링 및 로그 변환(log1p)
       - category_name을 대/중/소로 분해
       - 희귀 브랜드/카테고리를 'Other_*'로 통합 (rare category collapsing)
       - 결측 텍스트 채우기 및 텍스트 길이 기반 수치 피처 생성

    2) 텍스트 벡터화 + 차원 축소
       - simple_normalize()로 텍스트 정규화 후 *_clean 컬럼 생성
       - (2순위) name_for_vec / desc_for_vec 로 앞부분 일부만 사용
       - TF-IDF + n-gram((1,2)) + TruncatedSVD로 dense 벡터 생성
       - 카테고리/브랜드/배송 + 텍스트 길이 피처를 그대로 붙여서 최종 피처 구성

    3) PyCaret setup & compare_models
       - PyCaret 3.2.2 사용
       - 이미 price를 로그 변환했으므로 transformation=False
       - (1순위) dev 셋(예: 20만 행)만 사용해서 setup / compare_models 실행 가능
       - (1순위) compare_models 에서 가벼운 선형 계열 모델만 include

    4) 모델 성능 저장
       - predict_model() 결과의 Label(로그 스케일)을 expm1으로 되돌려
         원래 가격 스케일에서 R2 / RMSE / MAE 계산 후 JSON 저장
       - ⚠️ 이전 코드의 "predict_model() 안에 R2/RMSE/MAE 컬럼 있다고 가정"하던 오류 수정

    5) Test 예측 & submission 생성
       - test_vectorized에 대해 predict_model
       - 로그 예측값을 expm1으로 되돌려 price로 저장

    6) (3순위) PyCaret 바깥 Ridge baseline
       - sklearn Ridge 로 간단 baseline 모델 학습/평가

    7) 🔸 안전 하이퍼파라미터 튜닝(tune_best_model)
       - full 데이터를 그대로 tune_model에 넣으면 메모리 폭발 위험이 크므로
         내부에서 다시 "튜닝 전용 소형 PyCaret 세션"을 만들고,
         최대 safe_max_rows 행만 사용해서 ridge 하이퍼파라미터를 튜닝한다.
    """

    def __init__(
        self,
        data_dir="../data",
        images_dir="../images",
        results_dir="../results",
    ):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        self.train = None
        self.test = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        # 1순위: dev 셋 인덱스 / 3순위: Ridge baseline 보관
        self.dev_idx = None
        self.ridge_model = None

        os.makedirs(self.images_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 희귀 카테고리/브랜드를 "Other" 그룹으로 묶는 함수
    # ------------------------------------------------------------------
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        """
        col: 희귀 값 통합 대상 컬럼명 (예: 'brand_name', 'main_cat', ...)
        top_k: 등장 빈도 기준 상위 몇 개만 유지할지
        rare_label: 나머지(희귀값)를 치환할 문자열
        """
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()

        # 상위 top_k 값만 유지하고, 나머지는 rare_label로 통합
        top_values = value_counts.index[:top_k]

        self.train[col] = self.train[col].where(
            self.train[col].isin(top_values), rare_label
        )
        self.test[col] = self.test[col].where(
            self.test[col].isin(top_values), rare_label
        )

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 텍스트 간단 정규화 함수
    # ------------------------------------------------------------------
    def _simple_normalize(self, text: str) -> str:
        """
        Mercari 상위 솔루션 철학을 단순화한 텍스트 정규화 함수
        - 모두 소문자로 변환
        - 특정 특수문자(_ - . /)를 공백으로 치환
        - 숫자는 전부 'num' 토큰으로 치환
        - 다중 공백 정리
        """
        text = str(text).lower()
        # 특수문자 → 공백
        text = re.sub(r"[_\-\./]", " ", text)
        # 숫자를 'num'으로
        text = re.sub(r"\d+", " num ", text)
        # 공백 정리
        text = re.sub(r"\s+", " ", text).strip()
        return text

    # ------------------------------------------------------------------
    # 🔹 (공통 유틸) 텍스트 길이 제한 (2순위)
    # ------------------------------------------------------------------
    def _cut_text(self, text, max_words: int) -> str:
        """
        벡터화 비용 절감을 위해 앞 max_words 단어까지만 남기고 잘라냄.
        예:
          - name_for_vec  : max_words=20
          - desc_for_vec  : max_words=50
        """
        text = str(text)
        words = text.split()
        if len(words) <= max_words:
            return text
        return " ".join(words[:max_words])

    # ------------------------------------------------------------------
    # 1. 데이터 로딩 + 기본 전처리 + 희귀 카테고리/브랜드 통합 + 길이 피처 생성
    # ------------------------------------------------------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t"):
        print("📂 데이터 로딩 시작...")
        train_path = os.path.join(self.data_dir, train_file)
        test_path = os.path.join(self.data_dir, test_file)

        self.train = pd.read_csv(train_path, sep=sep)
        self.test = pd.read_csv(test_path, sep=sep)

        print(f"original data shape : train {self.train.shape}, test {self.test.shape}")
        print("✅ Price 외 결측치 처리 및 데이터 전처리 시작...")

        # --------------------------------------------------------------
        # 1-1) price 0 제거 + NaN 제거 (train만 해당)
        # --------------------------------------------------------------
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        print("Price NaN count:", self.train["price"].isna().sum())

        # --------------------------------------------------------------
        # 1-2) category_name → main_cat / sub_cat / sub_sub_cat 분해,
        #      텍스트 결측치 채우기, name 결측도 방지
        # --------------------------------------------------------------
        for df_name, df in [("train", self.train), ("test", self.test)]:
            # category_name이 "대/중/소" 형태이면 split, 아니면 'missing'
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (
                        x.split("/")
                        if isinstance(x, str) and "/" in x
                        else ["missing", "missing", "missing"]
                    )
                )
            )

            # 주요 텍스트 컬럼 결측치 채우기 + 문자열로 강제
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["category_name"] = df["category_name"].fillna("Unknown").astype(str)
            df["item_description"] = (
                df["item_description"].fillna("No description").astype(str)
            )
            df["name"] = df["name"].fillna("No name").astype(str)

            # train / test에 다시 반영 + 인덱스 리셋
            if df_name == "train":
                self.train = df.reset_index(drop=True)
            else:
                self.test = df.reset_index(drop=True)

        # --------------------------------------------------------------
        # 1-3) price 로그 변환 (log1p) - PyCaret 내부 transformation은 끔
        # --------------------------------------------------------------
        self.train["price"] = np.log1p(self.train["price"])

        # --------------------------------------------------------------
        # 1-4) 희귀 브랜드 / 카테고리 통합 (rare category collapsing)
        #      - 상위 N개만 유지, 나머지는 'Other_*'로 묶기
        # --------------------------------------------------------------
        print("📊 희귀 카테고리/브랜드 통합(rare category collapsing) 시작...")
        self._collapse_rare_values("brand_name", top_k=4500, rare_label="Other_brand")
        self._collapse_rare_values("main_cat", top_k=1000, rare_label="Other_main")
        self._collapse_rare_values("sub_cat", top_k=1000, rare_label="Other_sub")
        self._collapse_rare_values(
            "sub_sub_cat", top_k=1000, rare_label="Other_sub_sub"
        )

        # --------------------------------------------------------------
        # 1-5) 텍스트 길이 기반 수치 피처 추가
        #      - name_len_char / name_len_word
        #      - desc_len_char / desc_len_word
        # --------------------------------------------------------------
        for df_name, df in [("train", self.train), ("test", self.test)]:
            df["name_len_char"] = df["name"].astype(str).str.len()
            df["name_len_word"] = df["name"].astype(str).str.split().str.len()
            df["desc_len_char"] = df["item_description"].astype(str).str.len()
            df["desc_len_word"] = (
                df["item_description"].astype(str).str.split().str.len()
            )

        # --------------------------------------------------------------
        # 1-6) (2순위) 벡터화용 텍스트 길이 제한 컬럼 생성
        #      - name_for_vec : name의 앞 20단어
        #      - desc_for_vec : item_description의 앞 50단어
        # --------------------------------------------------------------
        for df in [self.train, self.test]:
            df["name_for_vec"] = df["name"].apply(lambda x: self._cut_text(x, 20))
            df["desc_for_vec"] = df["item_description"].apply(
                lambda x: self._cut_text(x, 50)
            )

        # --------------------------------------------------------------
        # 1-7) shipping / item_condition_id를 category 타입으로 캐스팅
        #      (PyCaret에 categorical_features로 넘길 예정)
        # --------------------------------------------------------------
        for df in [self.train, self.test]:
            df["shipping"] = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        print("Final Price NaN count:", self.train["price"].isna().sum())
        print("Train length:", len(self.train))
        print("\nTrain head:")
        print(self.train.head())

        print(f"\nTrain info:\n{'='*50}")
        print(self.train.info())

        print(
            f"\n✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}"
        )

    # ------------------------------------------------------------------
    # 2. 텍스트 벡터화 + 차원 축소 + 카테고리/길이 피처 결합
    # ------------------------------------------------------------------
    def vectorize_text(
        self,
        text_columns=["name_for_vec", "desc_for_vec"],  # ✅ 2순위: 잘라낸 텍스트 사용
        method="tfidf",
        max_features=50000,
        n_components=100,
    ):
        """
        text_columns: 벡터화할 텍스트 컬럼 리스트
                      (기본: name_for_vec, desc_for_vec)
        method: 'tfidf' 또는 'count'
        max_features: Vectorizer의 최대 피처 수
        n_components: TruncatedSVD 차원 수
        """
        print("📝 텍스트 벡터화 및 차원 축소 시작...")

        # --------------------------------------------------------------
        # 2-1) 텍스트 정규화 컬럼(*_clean) 생성
        #      - simple_normalize() 적용
        # --------------------------------------------------------------
        for col in text_columns:
            clean_col = f"{col}_clean"
            if clean_col not in self.train.columns:
                self.train[clean_col] = self.train[col].astype(str).apply(
                    self._simple_normalize
                )
                self.test[clean_col] = self.test[col].astype(str).apply(
                    self._simple_normalize
                )

        vectors = []
        feature_names = []

        # --------------------------------------------------------------
        # 2-2) 텍스트 컬럼별로 TF-IDF (또는 Count) + SVD 적용
        #      - ngram_range=(1,2)로 1~2그램 사용
        # --------------------------------------------------------------
        for col in tqdm(text_columns, desc="Text columns"):
            print(f"▶ 컬럼: {col}")
            clean_col = f"{col}_clean"

            if method == "tfidf":
                vec = TfidfVectorizer(
                    max_features=max_features,
                    ngram_range=(1, 2),
                )
            elif method == "count":
                vec = CountVectorizer(
                    max_features=max_features,
                    ngram_range=(1, 2),
                )
            else:
                raise ValueError("method must be 'tfidf' or 'count'")

            # train + test를 합쳐서 fit 후, 다시 train/test로 나눠 transform
            combined_text = pd.concat(
                [self.train[clean_col], self.test[clean_col]], axis=0
            )
            vec.fit(combined_text)

            train_vec = vec.transform(self.train[clean_col])
            test_vec = vec.transform(self.test[clean_col])

            # ----------------------------------------------------------
            # 2-3) 차원 축소: TruncatedSVD
            #      - TF-IDF의 차원이 max_features보다 작으면 SVD 생략
            # ----------------------------------------------------------
            if n_components < train_vec.shape[1]:
                svd = TruncatedSVD(n_components=n_components, random_state=23)
                train_vec = svd.fit_transform(train_vec)
                test_vec = svd.transform(test_vec)
                print(f"   ▪ 차원 축소 완료: {train_vec.shape[1]} components")
            else:
                train_vec = train_vec.toarray()
                test_vec = test_vec.toarray()

            vectors.append((train_vec, test_vec))
            feature_names.append([f"{col}_{i}" for i in range(train_vec.shape[1])])

            # 메모리 해제
            del combined_text, vec
            gc.collect()

        # --------------------------------------------------------------
        # 2-4) 모든 텍스트 피처를 가로 방향으로 합치기
        # --------------------------------------------------------------
        train_features = np.hstack([v[0] for v in vectors])
        test_features = np.hstack([v[1] for v in vectors])

        self.train_vectorized = pd.DataFrame(
            train_features, columns=[f for sub in feature_names for f in sub]
        )
        self.test_vectorized = pd.DataFrame(
            test_features, columns=[f for sub in feature_names for f in sub]
        )

        # --------------------------------------------------------------
        # 2-5) 카테고리/브랜드/배송 + 텍스트 길이 피처를 그대로 붙이기
        #      (LabelEncoder 제거 -> PyCaret 내부 인코딩 사용)
        # --------------------------------------------------------------
        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]

        numeric_length_cols = [
            "name_len_char",
            "name_len_word",
            "desc_len_char",
            "desc_len_word",
        ]

        for col in categorical_cols + numeric_length_cols:
            if col in self.train.columns:
                self.train_vectorized[col] = (
                    self.train[col].reset_index(drop=True)
                )
                self.test_vectorized[col] = (
                    self.test[col].reset_index(drop=True)
                )

        print(
            f"✅ 벡터화 + 차원 축소 + 카테고리/길이 피처 추가 완료: "
            f"train {self.train_vectorized.shape}, test {self.test_vectorized.shape}"
        )

    # ------------------------------------------------------------------
    # 3. PyCaret setup (1순위: dev 셋 사용 옵션)
    # ------------------------------------------------------------------
    def setup_pycaret(self, session_id=23, use_dev=True, dev_size=200_000):
        """
        1순위:
        - 전체 train_vectorized 대신 dev 셋(예: 200,000행)을 샘플링해서
          PyCaret setup / compare_models 에 사용.
        - 메모리 사용량과 실행 시간을 크게 줄이는 역할.

        use_dev=True 이면:
          - dev_size 만큼 train_vectorized에서 무작위 샘플링
          - self.dev_idx 에 dev 인덱스를 저장
        use_dev=False 이면:
          - 전체 train_vectorized 사용 (메모리 많이 사용)
        """
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        print("🔧 PyCaret setup 시작...")

        n_rows = len(self.train_vectorized)

        if use_dev:
            dev_size = min(dev_size, n_rows)
            rng = np.random.RandomState(session_id)
            self.dev_idx = rng.choice(n_rows, size=dev_size, replace=False)

            # ✅ dev 행만 뽑은 뒤, 인덱스를 0 ~ dev_size-1 로 초기화
            data_for_setup = self.train_vectorized.iloc[self.dev_idx].copy()
            target_for_setup = self.train["price"].iloc[self.dev_idx].copy()

            data_for_setup = data_for_setup.reset_index(drop=True)
            target_for_setup = target_for_setup.reset_index(drop=True)

            print(f"✅ dev 셋 사용: {dev_size} / {n_rows} 행으로 PyCaret setup")
        else:
            self.dev_idx = None
            data_for_setup = self.train_vectorized.copy()
            target_for_setup = self.train["price"].copy()

            # ✅ 전체 사용하는 경우에도 인덱스를 맞춰두면 안전
            data_for_setup = data_for_setup.reset_index(drop=True)
            target_for_setup = target_for_setup.reset_index(drop=True)

            print("⚠️ 전체 train_vectorized를 사용하여 PyCaret setup (메모리 주의)")

        # PyCaret에 넘길 카테고리 피처 이름들
        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]
        existing_categorical = [
            col for col in categorical_cols if col in data_for_setup.columns
        ]

        # 이미 price를 log1p로 변환했기 때문에 transformation=False로 설정
        self.setup_result = setup(
            data=data_for_setup.assign(price=target_for_setup),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            verbose=True,
        )
        print("✅ PyCaret setup 완료")

    # ------------------------------------------------------------------
    # 4. Base model 탐색 (1순위: 가벼운 모델만 비교)
    # ------------------------------------------------------------------
    def find_base_model(self, sort_metric="R2", fold=3, model_list=None):
        """
        1순위:
        - compare_models에 모든 모델을 넣으면 Mercari 데이터에서는
          메모리/시간 폭발 위험이 큼.
        - 기본값으로 선형 계열 가벼운 모델 위주로 include 목록을 제한.

        기본 include 목록:
          ['ridge', 'lasso', 'en', 'huber', 'par']
          (PyCaret model ID: Elastic Net 은 'en' 이 맞음)
        """
        if self.setup_result is None:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        print("🔍 Base model 탐색 시작...")

        if model_list is None:
            include_models = ["ridge", "lasso", "en", "huber", "par"]
        else:
            include_models = model_list

        self.best_model = compare_models(
            sort=sort_metric,
            n_select=1,
            include=include_models,
            fold=fold,
        )
        print(f"🏆 Best model 선택 완료: {self.best_model}")
        return self.best_model

    # ------------------------------------------------------------------
    # 🔸 4-1. 안전 하이퍼파라미터 튜닝
    # ------------------------------------------------------------------
    def tune_best_model(
        self,
        optimize_metric="R2",
        fold=3,
        n_iter=20,
        safe_max_rows=200_000,
        random_state=23,
        model_id="ridge",
    ):
        """
        full 데이터를 그대로 tune_model에 넣으면 메모리/시간이 폭발하므로,
        내부에서 다시 "튜닝 전용 소형 PyCaret 세션"을 만들어서 튜닝만 수행한다.

        - train_vectorized 에서 최대 safe_max_rows 행까지만 랜덤 샘플링
        - 해당 샘플로 setup → create_model(model_id) → tune_model
        - 튜닝된 모델을 self.best_model 에 저장

        기본값:
          - model_id = 'ridge' (지금까지 ridge가 항상 1등이었기 때문)
          - safe_max_rows = 200_000
          - n_iter = 20
        """
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        n_rows = len(self.train_vectorized)
        rng = np.random.RandomState(random_state)

        # 1) 튜닝에 사용할 행 인덱스 선택 (최대 safe_max_rows)
        if n_rows > safe_max_rows:
            idx = rng.choice(n_rows, size=safe_max_rows, replace=False)
            print(
                f"✅ 안전 튜닝 모드: 전체 {n_rows}행 중 {safe_max_rows}행만 랜덤 샘플링해서 사용합니다."
            )
        else:
            idx = np.arange(n_rows)
            print(
                f"✅ 안전 튜닝 모드: 데이터가 {n_rows}행이므로 전체를 사용해도 safe_max_rows({safe_max_rows}) 이하입니다."
            )

        X_tune = self.train_vectorized.iloc[idx].reset_index(drop=True)
        y_tune = self.train["price"].iloc[idx].reset_index(drop=True)

        # 2) 새 PyCaret 세션에서 setup (튜닝 전용)
        categorical_cols = [
            "main_cat",
            "sub_cat",
            "sub_sub_cat",
            "brand_name",
            "item_condition_id",
            "shipping",
        ]
        existing_categorical = [c for c in categorical_cols if c in X_tune.columns]

        print("🔧 [튜닝 전용] PyCaret setup 시작...")
        setup(
            data=X_tune.assign(price=y_tune),
            target="price",
            session_id=random_state,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            verbose=True,
        )
        print("✅ [튜닝 전용] PyCaret setup 완료")

        # 3) 선택한 모델 생성 후 하이퍼파라미터 튜닝
        print(f"🔍 '{model_id}' 모델 생성 및 튜닝 시작...")
        base_model = create_model(model_id, fold=fold)
        tuned_model = tune_model(
            base_model,
            optimize=optimize_metric,
            n_iter=n_iter,
            fold=fold,
        )
        print(f"🏁 튜닝 완료: {tuned_model}")

        # 4) 튜닝된 모델을 analyzer.best_model 에 저장
        self.best_model = tuned_model
        print("✅ self.best_model 을 튜닝된 모델로 업데이트했습니다.")
        return tuned_model

    # ------------------------------------------------------------------
    # 5. 모델 성능 저장 (원래 가격 스케일에서 R2/RMSE/MAE 계산)
    # ------------------------------------------------------------------
    def save_metrics(self, metrics_dict=None, model_name=None):
        """
        ⚠️ 수정된 내용:
        - PyCaret 3.2.2에서는 predict_model() 예측 컬럼명이 'prediction_label' 이므로,
          버전에 따라 'Label' 또는 'prediction_label' 둘 다 대응하기 위해
          _get_pred_column_name() 헬퍼를 사용한다.
        - price는 log1p(price_real)이므로, expm1()으로 되돌려 실제 가격 스케일에서
          R2 / RMSE / MAE를 계산한다.
        """
        if metrics_dict is None:
            if self.best_model is None:
                raise ValueError("모델이 없습니다.")

            # train 전체에 대해 예측 수행
            pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())

            # ✅ 버전에 따라 예측 컬럼 이름 찾기
            pred_col = self._get_pred_column_name(pred_df)

            # 로그 스케일 타깃/예측
            y_log_true = self.train["price"].values
            y_log_pred = pred_df[pred_col].values

            # expm1으로 원래 가격 스케일로 되돌리기
            y_true = np.expm1(y_log_true)
            y_pred = np.expm1(y_log_pred)

            r2 = r2_score(y_true, y_pred)
            rmse = mean_squared_error(y_true, y_pred, squared=False)
            mae = mean_absolute_error(y_true, y_pred)

            metrics_dict = {
                "R2": round(r2, 4),
                "RMSE": round(rmse, 4),
                "MAE": round(mae, 4),
            }

        self.metrics = metrics_dict

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        if model_name is None:
            model_name = str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )

        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장 완료: {file_path}")

    # ------------------------------------------------------------------
    # 6. 시각화
    # ------------------------------------------------------------------
    def visualize_model(self, plots=["residuals", "feature"]):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model() 또는 tune_best_model()로 모델을 선택하세요.")

        print("🎨 시각화 시작...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = str(self.best_model).split("(")[0]

        for p in plots:
            try:
                # feature_importance 같은 alias를 쓰더라도 PyCaret plot 이름에 맞게 매핑
                plot_name = "feature" if p == "feature_importance" else p

                save_path = os.path.join(
                    self.images_dir, f"{model_name}_{plot_name}_{timestamp}.png"
                )
                plot_model(self.best_model, plot=plot_name, save=True)
                print(f"✅ {plot_name} plot 저장 완료: {save_path}")
            except Exception as e:
                print(f"⚠️ Plot {p} 실패: {e}")

    # ------------------------------------------------------------------
    # 7. Test 예측 & submission 생성
    # ------------------------------------------------------------------
    def predict_test(self, submission_file="submission.csv"):
        if self.best_model is None:
            raise ValueError("먼저 find_base_model() 또는 tune_best_model()로 모델을 선택하세요.")

        print("📦 Test 데이터 예측 시작...")

        # test_vectorized에 대해 예측 (로그 스케일)
        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())

        # ✅ 버전에 따라 예측 컬럼 이름 찾기
        pred_col = self._get_pred_column_name(predictions)

        # 로그 예측값 → 실제 가격
        price_log_pred = predictions[pred_col].values
        price_pred = np.expm1(price_log_pred)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": price_pred}
        )

        submission_path = os.path.join(self.results_dir, submission_file)
        submission.to_csv(submission_path, index=False)
        print(f"💾 Submission 저장 완료: {submission_path}")
        return submission

    # ------------------------------------------------------------------
    # 8. 3순위: Ridge baseline 학습 (PyCaret 바깥)
    # ------------------------------------------------------------------
    def train_ridge_baseline(self, alpha=1.0, use_dev=False):
        """
        3순위:
        - PyCaret AutoML과 별도로, sklearn Ridge로 간단한 baseline 모델을 학습.
        - dev 셋 또는 전체 데이터 중 선택 가능.

        alpha: Ridge 정규화 강도
        use_dev=True  이고 self.dev_idx가 존재하면 dev 셋만 사용
        use_dev=False 이면 전체 train_vectorized 사용
        """
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        if use_dev and self.dev_idx is not None:
            idx = self.dev_idx
            print(f"🔹 Ridge baseline: dev 셋 {len(idx)} 행으로 학습")
        else:
            idx = np.arange(len(self.train_vectorized))
            print(f"🔹 Ridge baseline: 전체 {len(idx)} 행으로 학습 (메모리 주의)")

        X = self.train_vectorized.iloc[idx].copy()
        y = self.train["price"].iloc[idx].values  # 로그 스케일 타깃

        # Ridge는 숫자형만 받으므로 object/category는 코드로 변환
        non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
        for col in non_numeric_cols:
            X[col] = X[col].astype("category").cat.codes

        X_values = X.values

        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_values, y)

        self.ridge_model = model
        print("✅ Ridge baseline 학습 완료")

        return model

    # ------------------------------------------------------------------
    # 9. Ridge baseline 평가 (선택)
    # ------------------------------------------------------------------
    def evaluate_ridge_baseline(self, use_dev=False):
        """
        3순위:
        - 학습된 Ridge baseline 성능을 원래 가격 스케일에서 평가.
        - dev 셋 또는 전체 데이터 기준 선택.
        """
        if self.ridge_model is None:
            raise ValueError("먼저 train_ridge_baseline()으로 모델을 학습하세요.")
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        if use_dev and self.dev_idx is not None:
            idx = self.dev_idx
            print(f"🔹 Ridge baseline 평가: dev 셋 {len(idx)} 행 사용")
        else:
            idx = np.arange(len(self.train_vectorized))
            print(f"🔹 Ridge baseline 평가: 전체 {len(idx)} 행 사용")

        X = self.train_vectorized.iloc[idx].copy()
        y_log_true = self.train["price"].iloc[idx].values

        non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
        for col in non_numeric_cols:
            X[col] = X[col].astype("category").cat.codes

        X_values = X.values
        y_log_pred = self.ridge_model.predict(X_values)

        y_true = np.expm1(y_log_true)
        y_pred = np.expm1(y_log_pred)

        r2 = r2_score(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mae = mean_absolute_error(y_true, y_pred)

        print(
            f"🔎 Ridge baseline 성능 - R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}"
        )

        return {"R2": r2, "RMSE": rmse, "MAE": mae}

    # ------------------------------------------------------------------
    # (공통 유틸) PyCaret predict_model 결과에서 예측 컬럼 이름 찾기
    # ------------------------------------------------------------------
    def _get_pred_column_name(self, df: pd.DataFrame) -> str:
        """
        PyCaret 2.x / 3.x 버전 차이를 흡수하기 위한 헬퍼.
        - 일부 버전: 예측 컬럼명이 'Label'
        - PyCaret 3.x: 예측 컬럼명이 'prediction_label'
        둘 중 하나를 자동으로 찾아서 반환하고,
        둘 다 없으면 에러를 발생시킨다.
        """
        if "Label" in df.columns:
            return "Label"
        if "prediction_label" in df.columns:
            return "prediction_label"
        raise ValueError(
            f"predict_model 결과에서 예측 컬럼을 찾을 수 없습니다. "
            f"컬럼들: {list(df.columns)}"
        )


In [15]:
# ----------------------------------------------------------------------
# 실행 예시
# ----------------------------------------------------------------------
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file="train.tsv", test_file="test.tsv")
# analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)
# analyzer.setup_pycaret(use_dev=False)         # 또는 use_dev=True, dev_size=200_000
# analyzer.find_base_model(sort_metric="R2")
# analyzer.tune_best_model(optimize_metric="R2", fold=3)  # ✅ 새로 추가된 튜닝 단계
# analyzer.save_metrics()
# analyzer.visualize_model(plots=["residuals", "feature"])
# analyzer.predict_test(submission_file="submission.csv")
# analyzer.train_ridge_baseline(alpha=1.0, use_dev=True)
# analyzer.evaluate_ridge_baseline(use_dev=True)

In [2]:
analyzer = MercariPyCaretAnalyzer()

In [3]:
analyzer.load_data(train_file="train.tsv", test_file="test.tsv")

📂 데이터 로딩 시작...
original data shape : train (1482535, 8), test (693359, 7)
✅ Price 외 결측치 처리 및 데이터 전처리 시작...
Price NaN count: 0
📊 희귀 카테고리/브랜드 통합(rare category collapsing) 시작...
Final Price NaN count: 0
Train length: 1481661

Train head:
   train_id                                 name item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                 3   
1         1     Razer BlackWidow Chroma Keyboard                 3   
2         2                       AVA-VIV Blouse                 1   
3         3                Leather Horse Statues                 1   
4         4                 24K GOLD plated rose                 1   

                                       category_name brand_name     price  \
0                                  Men/Tops/T-shirts    Unknown  2.397895   
1  Electronics/Computers & Tablets/Components & P...      Razer  3.970292   
2                        Women/Tops & Blouses/Blouse     Target  2.397895   
3                 Home/Home Décor/Ho

In [4]:
# 기본값: name_for_vec / desc_for_vec 사용, max_features / n_components 는 팀원 코드에 맞춰 조정
analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)

📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:   0%|          | 0/2 [00:00<?, ?it/s]

▶ 컬럼: name_for_vec


Text columns:  50%|█████     | 1/2 [01:49<01:49, 109.23s/it]

   ▪ 차원 축소 완료: 100 components
▶ 컬럼: desc_for_vec


Text columns: 100%|██████████| 2/2 [06:39<00:00, 199.54s/it]

   ▪ 차원 축소 완료: 100 components


✅ 벡터화 + 차원 축소 + 카테고리/길이 피처 추가 완료: train (1481661, 210), test (693359, 210)


In [5]:
# ✅ 추천 플로우: 전체 train 으로 AutoML (가장 싸게 성능 끌어올리는 방법)
analyzer.setup_pycaret(use_dev=False)     # 기본값이라 analyzer.setup_pycaret() 만 써도 됨

🔧 PyCaret setup 시작...
⚠️ 전체 train_vectorized를 사용하여 PyCaret setup (메모리 주의)


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(1481661, 211)"
4,Transformed data shape,"(1481661, 225)"
5,Transformed train set shape,"(1037162, 225)"
6,Transformed test set shape,"(444499, 225)"
7,Numeric features,204
8,Categorical features,6
9,Preprocess,True


✅ PyCaret setup 완료


In [6]:
analyzer.find_base_model(sort_metric="R2") # 내부에서 ridge/lasso/en/huber/par만 비교

🔍 Base model 탐색 시작...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ridge,Ridge Regression,0.4250,0.3099,0.5567,0.4436,0.1384,0.1508,21.8133
huber,Huber Regressor,0.4232,0.3117,0.5583,0.4404,0.1381,0.1483,39.2967
lasso,Lasso Regression,0.5819,0.5570,0.7463,-0.0000,0.1850,0.2098,16.6367
en,Elastic Net,0.5819,0.5570,0.7463,-0.0000,0.1850,0.2098,16.2667
par,Passive Aggressive Regressor,0.6350,0.6725,0.8197,-0.2071,0.2090,0.2252,16.2667


🏆 Best model 선택 완료: Ridge(random_state=23)


Ridge(random_state=23)

In [7]:
analyzer.tune_best_model(optimize_metric="R2", fold=3, n_iter=20)  # ✅ 새로 추가된 튜닝 단계
# 이미 뽑힌 best model(Ridge)을 그대로 쓰지 말고 tune_model로 하이퍼파라미터를 튜닝해서 쓰는 것이야.
# 우리 피처 수(200개 조금 넘는 수준) + 선형모델이라서, 튜닝 비용도 아주 작고
# 캐글 상위권들처럼 alpha(정규화 강도)를 잘 잡는 것만으로도 성능이 꽤 올라가는 경우

✅ 안전 튜닝 모드: 전체 1481661행 중 200000행만 랜덤 샘플링해서 사용합니다.
🔧 [튜닝 전용] PyCaret setup 시작...


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(200000, 211)"
4,Transformed data shape,"(200000, 225)"
5,Transformed train set shape,"(140000, 225)"
6,Transformed test set shape,"(60000, 225)"
7,Numeric features,204
8,Categorical features,6
9,Preprocess,True


✅ [튜닝 전용] PyCaret setup 완료
🔍 'ridge' 모델 생성 및 튜닝 시작...


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.4274,0.3136,0.5600,0.4298,0.1393,0.1520
1,0.4318,0.3207,0.5663,0.4199,0.1408,0.1534
2,0.4269,0.3116,0.5582,0.4324,0.1389,0.1515
Mean,0.4287,0.3153,0.5615,0.4273,0.1397,0.1523
Std,0.0022,0.0039,0.0035,0.0054,0.0008,0.0008


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.4274,0.3136,0.5600,0.4298,0.1393,0.1520
1,0.4318,0.3207,0.5663,0.4199,0.1408,0.1534
2,0.4269,0.3116,0.5582,0.4324,0.1389,0.1515
Mean,0.4287,0.3153,0.5615,0.4273,0.1397,0.1523
Std,0.0022,0.0039,0.0035,0.0054,0.0008,0.0008


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
🏁 튜닝 완료: Ridge(random_state=23)
✅ self.best_model 을 튜닝된 모델로 업데이트했습니다.


Ridge(random_state=23)

In [8]:
# PyCaret best model 기반 평가/시각화/예측
analyzer.save_metrics()

💾 Metrics 저장 완료: ../results\Ridge_metrics_20251204_171808.json


In [9]:
analyzer.visualize_model(plots=["residuals", "feature"])

🎨 시각화 시작...


✅ residuals plot 저장 완료: ../images\Ridge_residuals_20251204_171808.png


✅ feature plot 저장 완료: ../images\Ridge_feature_20251204_171808.png


In [10]:
analyzer.predict_test(submission_file="submission_tuned_ridge.csv")

📦 Test 데이터 예측 시작...


💾 Submission 저장 완료: ../results\submission_tuned_ridge.csv


,test_id,price
0,0,17.067796
1,1,10.762523
2,2,41.233594
3,3,17.060624
4,4,9.277059
...,...,...
693354,693354,16.337711
693355,693355,23.326971
693356,693356,5.657661
693357,693357,14.552038


In [11]:
# ---------------------------------------------------------
# 0) 전제: 이미 아래가 끝난 상태라고 가정
# ---------------------------------------------------------
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file="train.tsv", test_file="test.tsv")
# analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)
# analyzer.setup_pycaret(use_dev=False)  # 지금처럼 full 데이터 사용

from pycaret.regression import compare_models, blend_models

# ---------------------------------------------------------
# 1) 선형 계열 상위 3개 모델 뽑기
# ---------------------------------------------------------
linear_candidates = ["ridge", "huber", "lasso", "en", "par"]

top_models = compare_models(
    sort="R2",
    include=linear_candidates,
    n_select=3,   # 상위 3개만 앙상블에 사용
    fold=3,
)

print("\n[선택된 상위 3개 모델]")
for m in top_models:
    print(" -", m)


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
ridge,Ridge Regression,0.4287,0.3153,0.5615,0.4273,0.1397,0.1523,3.4433
huber,Huber Regressor,0.4269,0.3174,0.5633,0.4236,0.1394,0.1497,5.9867
lasso,Lasso Regression,0.5795,0.5506,0.7421,-0.0000,0.1844,0.2095,3.1633
en,Elastic Net,0.5795,0.5506,0.7421,-0.0000,0.1844,0.2095,3.1267
par,Passive Aggressive Regressor,0.6437,0.6800,0.8245,-0.2351,0.2135,0.2294,2.4667



[선택된 상위 3개 모델]
 - Ridge(random_state=23)
 - HuberRegressor()
 - Lasso(random_state=23)


In [14]:
import numpy as np
import pandas as pd
from pycaret.regression import predict_model
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def ensemble_predict_log(models, X_frame: pd.DataFrame, analyzer) -> np.ndarray:
    """
    models   : PyCaret로 학습/튜닝한 모델 객체 리스트 (ex. top_models)
    X_frame  : analyzer.train_vectorized 또는 analyzer.test_vectorized 같은 피처 DataFrame
    analyzer : MercariPyCaretAnalyzer 인스턴스 (예: analyzer)

    반환값   : log1p(price) 스케일에서의 앙상블 예측 벡터
    """
    preds = []

    for m in models:
        # 원본 DataFrame 망가지는 걸 방지하려고 copy() 사용
        df_pred = predict_model(m, data=X_frame.copy())

        # ✅ 클래스 안에 이미 있는 헬퍼를 그대로 사용
        pred_col = analyzer._get_pred_column_name(df_pred)

        preds.append(df_pred[pred_col].values)

    # 모델별 예측을 행 방향으로 쌓고, 열 방향(=모델 축) 평균
    avg_log_pred = np.mean(preds, axis=0)
    return avg_log_pred


def evaluate_ensemble_on_train(models, analyzer):
    """
    train 전체에 대해 수동 앙상블 성능을 계산해서 원래 가격 스케일 기준
    R2 / RMSE / MAE 를 출력해주는 유틸 함수.
    """
    # 1) 로그 스케일 타깃
    y_log_true = analyzer.train["price"].values

    # 2) 로그 스케일 앙상블 예측
    ens_log_pred = ensemble_predict_log(models, analyzer.train_vectorized, analyzer)

    # 3) 원래 가격으로 복원
    y_true = np.expm1(y_log_true)
    y_pred = np.expm1(ens_log_pred)

    # 4) 메트릭 계산
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)

    print("🔎 수동 앙상블 (train 기준) 성능")
    print(f"   R2   : {r2:.4f}")
    print(f"   RMSE : {rmse:.4f}")
    print(f"   MAE  : {mae:.4f}")

    return {"R2": r2, "RMSE": rmse, "MAE": mae}


def make_ensemble_submission(models, analyzer, filename="submission_ensemble_manual.csv"):
    """
    수동 앙상블로 test 예측을 해서 캐글 제출용 파일을 만들어 주는 함수.
    """
    # 1) test 에 대한 로그 스케일 앙상블 예측
    ens_log_test = ensemble_predict_log(models, analyzer.test_vectorized, analyzer)

    # 2) 원래 가격으로 복원
    price_pred = np.expm1(ens_log_test)

    # 3) submission DF 생성
    submission = pd.DataFrame({
        "test_id": analyzer.test["test_id"],
        "price": price_pred,
    })

    # 4) 저장
    save_path = os.path.join(analyzer.results_dir, filename)
    submission.to_csv(save_path, index=False)
    print(f"💾 수동 앙상블 submission 저장 완료: {save_path}")

    return submission

In [15]:
ens_metrics = evaluate_ensemble_on_train(top_models, analyzer)
ens_metrics

🔎 수동 앙상블 (train 기준) 성능
   R2   : 0.1358
   RMSE : 35.8770
   MAE  : 13.2656


{'R2': 0.13575056288281262,
 'RMSE': 35.877027482319974,
 'MAE': 13.2655813869565}

In [17]:
ens_metrics = evaluate_ensemble_on_train(top_models, analyzer)

# 소수점 4자리 정도로 정리 (선택 사항)
ens_metrics_rounded = {k: round(v, 4) for k, v in ens_metrics.items()}

analyzer.save_metrics(
    metrics_dict=ens_metrics_rounded,      # 👉 우리가 직접 계산한 앙상블 메트릭
    model_name="manual_ensemble_top3",    # 👉 파일 이름에 들어갈 태그
)

🔎 수동 앙상블 (train 기준) 성능
   R2   : 0.1358
   RMSE : 35.8770
   MAE  : 13.2656
💾 Metrics 저장 완료: ../results\manual_ensemble_top3_metrics_20251204_181317.json


In [ ]:
analyzer.visualize_model(plots=["residuals", "feature"])

In [ ]:
submission_ens = make_ensemble_submission(
    models=top_models,
    analyzer=analyzer,
    filename="submission_manual_ensemble.csv",
)

In [ ]:
from sklearn.metrics import mean_squared_log_error
import numpy as np

# y_true, y_pred: expm1 후의 원래 가격 스케일
rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred))
print("RMSLE:", rmsle)
